# Train your own ML model

A hands-on introduction to training a (convolutional) neural network. You will write three small pieces of code; the rest is ready to run. No GPU needed.

**What you'll learn:** 
- What a neural network actually looks like
- The steps in developing a model
- How you can evaluate model performance 

**Run it:** You can run it either locally or on Google Colab. 

Local: In this repo's root, run 

```bash
uv venv --python 3.11 .venv
.venv/Scripts/activate
uv pip install --python python tensorflow-cpu matplotlib ipykernel
```
In VSCode you can select the kernel in the top-right corner. Select `.venv/Scripts/python.exe` as its kernel.

Cloud: Go to [colab.research.google.com](https://colab.research.google.com/), choose **File > Upload notebook**, and run cells in order so you can stop at the questions. Choose a CPU runtime; no Colab Pro or Google Drive is needed. Colab includes TensorFlow, NumPy and Matplotlib. The first run downloads MNIST, so internet is required.

**Instructions:** In each TODO, replace the `raise NotImplementedError(...)` line, not the whole cell. The expandable hints give you valid Python to copy and adapt; opening the solutions is fine if you want to focus on the model rather than syntax. Run the cell after each change to get immediate feedback.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

SEED = 42
keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)

(images_pool, labels_pool), (test_images_raw, test_labels) = keras.datasets.mnist.load_data()
print('Training pool:', images_pool.shape, 'Test set:', test_images_raw.shape)
print('One image:', images_pool.shape[1:], images_pool.dtype)

## 1. Look at the data before fitting anything

Each input is a 28 x 28 array of brightness values from 0 to 255. Its label is an integer from 0 to 9, assigned by a person. Guess a few digits from the grid, then run the next cell to reveal the labels. Are any ambiguous even to you?

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(images_pool[i], cmap='gray', vmin=0, vmax=255)
    ax.set_title(f'Image {i + 1}')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
print('Top row labels:   ', labels_pool[:5].tolist())
print('Bottom row labels:', labels_pool[5:10].tolist())

## 2. Your turn: prepare the pixels

Neural networks train more predictably with small, consistently scaled numbers. Fill in `prepare_images`: turn integer pixels into `float32` values between 0 and 1. **Keep the 28 x 28 shape**; the model will flatten it later. The checks below tell you whether it worked.

<details><summary>Hint: Python template</summary>Replace the <code>raise</code> line with <code>return images.astype(np.float32) / 255.0</code>. <code>return</code> sends a new array back to the caller; <code>astype</code> converts the pixel type, and division scales 255 to 1.0.</details>
<details><summary>Solution, open only if stuck</summary><code>return images.astype(np.float32) / 255.0</code></details>

In [ ]:
def prepare_images(images):
    # TODO 1: Return float32 pixels scaled to [0, 1].
    raise NotImplementedError('Complete TODO 1 before continuing')

sample = np.array([[[0, 128, 255]]], dtype=np.uint8)
scaled = prepare_images(sample)
assert scaled.shape == sample.shape
assert scaled.dtype == np.float32
np.testing.assert_allclose(scaled, [[[0.0, 128 / 255, 1.0]]], rtol=1e-6)
assert sample[0, 0, 2] == 255  # The caller's data stays unchanged.
print('Preprocessing checks passed.')

### Three sets, three jobs

We take a **shuffled subset** of the official MNIST training pool to keep CPU training short. Training images update the weights. Validation images help us choose settings and notice overfitting; they never update weights. The official test set is held back until the final evaluation. Picking a model after looking at test errors makes the test score less honest.

Question for the group: What would happen if the same handwritten image appeared in both training and testing?

In [ ]:
rng = np.random.default_rng(SEED)
order = rng.permutation(len(images_pool))
train_idx, val_idx = order[:12_000], order[12_000:14_000]
x_train, y_train = prepare_images(images_pool[train_idx]), labels_pool[train_idx]
x_val, y_val = prepare_images(images_pool[val_idx]), labels_pool[val_idx]
x_test = prepare_images(test_images_raw)
assert not np.intersect1d(train_idx, val_idx).size
print('Train / validation / test:', len(x_train), len(x_val), len(x_test))
print('Always guess 0 baseline, test accuracy:', round(np.mean(test_labels == 0), 3))

## 3. Your turn: build a tiny model

Build a `keras.Sequential` model with `Input(shape=(28, 28))`, `Flatten()`, `Dense(64, activation='relu')`, and `Dense(10)` in that order. The hidden layer learns combinations of pixels; the 10 outputs are **logits**, one score per digit. We do not apply softmax in the model because the loss below expects raw logits.

With a partner, estimate how many trainable parameters this model has. Then check `model.summary()`.

<details><summary>Hint: Python template</summary>Replace the <code>raise</code> line with a <code>return keras.Sequential([...])</code> expression. Put these four entries between the square brackets, separated by commas: <code>keras.Input(shape=(28, 28))</code>, <code>keras.layers.Flatten()</code>, <code>keras.layers.Dense(64, activation='relu')</code>, <code>keras.layers.Dense(10)</code>. The 784 flattened inputs each connect to 64 hidden units; each hidden unit has a bias. The output layer has 64 inputs and 10 units.</details>
<details><summary>Solution, open only if stuck</summary><pre>return keras.Sequential([
    keras.Input(shape=(28, 28)),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(10),
])</pre></details>

In [ ]:
def build_model():
    # TODO 2: Return the Sequential model described above.
    raise NotImplementedError('Complete TODO 2 before continuing')


model = build_model()
assert model.input_shape == (None, 28, 28)
assert model.output_shape == (None, 10)
assert model.count_params() == 50_890
assert isinstance(model.layers[0], keras.layers.Flatten)
assert isinstance(model.layers[1], keras.layers.Dense) and model.layers[1].units == 64
assert model.layers[1].activation.__name__ == 'relu'
assert isinstance(model.layers[2], keras.layers.Dense) and model.layers[2].units == 10
print('Model checks passed.')
model.summary()

## 4. Let it learn

For each batch, the model predicts logits. Sparse cross-entropy compares those scores with the integer labels. Backpropagation computes how each weight affected the loss; Adam uses those gradients to update the weights. An epoch is one pass over the training subset. Watch **validation** as well as training performance.

Before running: Will the first epoch already beat the 'always 0' baseline?

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy')],
)
history = model.fit(
    x_train, y_train, batch_size=128, epochs=5,
    validation_data=(x_val, y_val), verbose=2,
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric in zip(axes, ['loss', 'accuracy']):
    ax.plot(history.history[metric], marker='o', label='train')
    ax.plot(history.history['val_' + metric], marker='o', label='validation')
    ax.set(xlabel='Epoch (starting at 0)', ylabel=metric, title=metric)
    ax.legend()
plt.tight_layout()
plt.show()
print('Last validation accuracy:', round(history.history['val_accuracy'][-1], 3))

Compare the curves. If training accuracy rises while validation accuracy stalls or falls, the model may be memorizing the training set. Five epochs may not show this clearly. Try more epochs **later**, using validation results to decide when to stop; do not use the test set to choose the epoch count.

## 5. Your turn: measure the mistakes

Only now evaluate on the untouched official test set. Accuracy is a single number; a **confusion matrix** tells us which digits get mixed up. Make a 10 x 10 integer array, where row `i` is the true digit and column `j` is the predicted digit. Each image contributes exactly one count.

<details><summary>Hint: Python template</summary>Replace the <code>raise</code> line with these three lines, indented four spaces: <code>matrix = np.zeros((10, 10), dtype=int)</code>, then <code>np.add.at(matrix, (truth, predicted), 1)</code>, then <code>return matrix</code>. <code>np.add.at</code> increments a cell for each pair, even when pairs repeat. The parentheses around <code>(truth, predicted)</code> make a tuple of row and column indices.</details>
<details><summary>Solution, open only if stuck</summary><pre>matrix = np.zeros((10, 10), dtype=int)
np.add.at(matrix, (truth, predicted), 1)
return matrix</pre></details>

In [ ]:
def confusion_counts(truth, predicted):
    # TODO 3: Return a 10 x 10 count matrix (rows: truth; columns: prediction).
    raise NotImplementedError('Complete TODO 3 before continuing')


example = confusion_counts(np.array([3, 3, 7]), np.array([3, 7, 3]))
assert example.shape == (10, 10) and np.issubdtype(example.dtype, np.integer)
assert example.sum() == 3 and example[3, 3] == 1
assert example[3, 7] == 1 and example[7, 3] == 1
print('Confusion matrix checks passed.')

In [ ]:
def show_test_errors(predicted, model_name, n_examples=10):
    counts = confusion_counts(test_labels, predicted)
    assert counts.shape == (10, 10) and counts.sum() == len(test_labels)
    assert np.array_equal(counts.sum(axis=1), np.bincount(test_labels, minlength=10))
    print(f'{model_name} correct: {np.trace(counts)} / {len(test_labels)}')

    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(counts, cmap='Blues')
    fig.colorbar(im, ax=ax, label='Number of images')
    ax.set(xticks=range(10), yticks=range(10), xlabel='Predicted', ylabel='Actual',
           title=f'{model_name} test-set confusion matrix')
    plt.tight_layout()
    plt.show()

    wrong = np.flatnonzero(predicted != test_labels)
    fig, axes = plt.subplots(2 if n_examples > 5 else 1, 5,
                             figsize=(10, 4 if n_examples > 5 else 2.5))
    for ax, idx in zip(axes.flat, wrong[:n_examples]):
        ax.imshow(test_images_raw[idx], cmap='gray', vmin=0, vmax=255)
        ax.set_title(f'True {test_labels[idx]}, guess {predicted[idx]}')
        ax.axis('off')
    for ax in list(axes.flat)[len(wrong[:n_examples]):]:
        ax.axis('off')
    fig.suptitle(f'{model_name} mistakes')
    plt.tight_layout()
    plt.show()
    return counts


test_loss, test_accuracy = model.evaluate(x_test, test_labels, verbose=0)
predictions = model.predict(x_test, batch_size=256, verbose=0).argmax(axis=1)
print(f'Test accuracy: {test_accuracy:.3f}; test loss: {test_loss:.3f}')
matrix = show_test_errors(predictions, 'Dense model')

# Some things to consider
- Do the digits from this dataset actually reflect how users would write numbers?
- Does the model overfit on the training data? 
- Which digits are the most difficult for our model?

When developing a ML model, these are things to be considered. 

If you prefer another hands-on experiment instead of the CNN below, try 1,000 training images, 128 hidden units, or 15 epochs. Change only one thing and predict its effect on validation accuracy. Restart and run through the learning-curve cell with the same split.

## Optional extension: try a small convolutional neural network (CNN)

As an extension, you can also try a CNN, which is the same type of model we use for the AQUAFIND project! It is a model type that is good at processing spatial information. But will it also help for the task of predicting handwritten digits? This is a ready-to-run demo, not another coding exercise.

### 1. Inspect the new input and model

Keras expects an image channel dimension: `(28, 28, 1)` for a grayscale digit. Before running this cell, guess whether this CNN will have more or fewer trainable weights than the 50,890-weight dense model. Then read `cnn_model.summary()` and identify where spatial dimensions shrink.

In [ ]:
cnn_train = x_train[..., np.newaxis]
cnn_val = x_val[..., np.newaxis]
cnn_test = x_test[..., np.newaxis]
assert cnn_train.shape == (len(x_train), 28, 28, 1)

keras.utils.set_random_seed(SEED)
cnn_model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    keras.layers.Conv2D(16, 3, activation='relu'),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(32, 3, activation='relu'),
    keras.layers.MaxPooling2D(),
    keras.layers.Flatten(),
    keras.layers.Dense(10),
])
cnn_model.summary()

### 2. Train on the same images

The training split, labels, learning rate, batch size and five epochs are unchanged. Which parts of the model can now recognize a stroke in different positions? Watch training and validation accuracy during fitting.

In [ ]:
cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy')],
)
cnn_history = cnn_model.fit(
    cnn_train, y_train, batch_size=128, epochs=5,
    validation_data=(cnn_val, y_val), verbose=2,
)

### 3. Compare validation results

Did the CNN improve throughout training, or only at the end? Compare the whole validation curve, not just a single score. The models use the same split and training schedule, but differ in architecture, training time and parameter count.

In [ ]:
print(f'Dense final validation accuracy: {history.history["val_accuracy"][-1]:.3f}')
print(f'CNN final validation accuracy:   {cnn_history.history["val_accuracy"][-1]:.3f}')
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history.history['val_accuracy'], marker='o', label='Dense')
ax.plot(cnn_history.history['val_accuracy'], marker='o', label='CNN')
ax.set(xlabel='Epoch (starting at 0)', ylabel='Validation accuracy',
       title='Same split, different spatial model')
ax.legend()
plt.tight_layout()
plt.show()

### 4. Look at test errors

Now check both models on the same test images. We already inspected the dense model's test errors, so this comparison illustrates the idea; it is not a fresh, unbiased test after model selection.

In [ ]:
cnn_test_loss, cnn_test_accuracy = cnn_model.evaluate(cnn_test, test_labels, verbose=0)
print(f'Dense test accuracy: {test_accuracy:.3f}; CNN test accuracy: {cnn_test_accuracy:.3f}')

First inspect the CNN's confusion matrix and five wrong guesses. Then compare both models' guesses on the *same* images. Where does each model rescue the other's mistake?

In [ ]:
cnn_predictions = cnn_model.predict(cnn_test, batch_size=256, verbose=0).argmax(axis=1)
cnn_matrix = show_test_errors(cnn_predictions, 'CNN', n_examples=5)

In [ ]:
dense_right = predictions == test_labels
cnn_right = cnn_predictions == test_labels
groups = [
    ('Dense right, CNN wrong', dense_right & ~cnn_right),
    ('CNN right, dense wrong', cnn_right & ~dense_right),
    ('Both wrong', ~dense_right & ~cnn_right),
]
for label, mask in groups:
    indices = np.flatnonzero(mask)
    print(f'{label}: {len(indices)} images')
    if not len(indices):
        continue
    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    fig.suptitle(f'{label} (first {min(2, len(indices))} in test-set order)')
    for ax, idx in zip(axes, indices[:2]):
        ax.imshow(test_images_raw[idx], cmap='gray', vmin=0, vmax=255)
        ax.set_title(f'True {test_labels[idx]}\nDense {predictions[idx]} | CNN {cnn_predictions[idx]}', fontsize=10)
        ax.axis('off')
    for ax in axes[len(indices[:2]):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## Takeaways

- The workflow matters as much as the model: inspect, preprocess, train, validate, then test.
- Preprocessing affects optimization; scaling pixels to `[0, 1]` makes training more predictable.
- Architecture encodes assumptions: dense layers ignore spatial relationships, while CNNs reuse local filters.
- Accuracy is only a summary. Confusion matrices and example mistakes reveal where models fail.
- The same workflow applies to AQUAFIND: match the architecture to the data, use validation for decisions, and reserve test data for final evaluation.
- A more advanced architecture is not automatically better. Data quality, labels, training choices, and meaningful metrics matter too.